# Model Training

Prepare the Training Dataset and experiment with different models for automatically predict the `Test Results` based on a list of patient's features.

# Setup Notebook

## Imports

In [1]:
# Import Standard Libraries
import os
import mlflow
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Import Package Modules
from src.general_utils.general_utils import read_configuration
from src.data_preparation.data_preparation import HealthcareDataPreparation
from src.model_training.model_training import ModelTrainer

ModuleNotFoundError: No module named 'pkg_resources'

## Define Configurations

In [ ]:
# Retrieve root path
root_path = Path(os.getcwd()).parents[1]

In [ ]:
# Read configuration variables
config = read_configuration(root_path / 
                            'configuration' / 
                            'healthcare_classification_config.yaml')

In [ ]:
# Extract configuration variables
# -------- Data Pipeline -------
dataset_config = config['healthcare_data_pipeline_config']['dataset']
data_transformations_config = config['healthcare_data_pipeline_config']['data_transformations']
features_config = config['healthcare_data_pipeline_config']['features']
labels_config = config['healthcare_data_pipeline_config']['labels']
train_test_split_config = config['healthcare_data_pipeline_config']['train_test_split']
# ------------------------------

# -------- Model Training -------
model_training_config = config['model_training_config']
# -------------------------------

# Read Data

## Healthcare Dataset

In [ ]:
# Define path
data_path = (root_path /
             dataset_config['data_path'][0] /
             dataset_config['data_path'][1] /
             dataset_config['data_path'][2])

In [ ]:
# Read data
dataset = pd.read_csv(data_path,
                      parse_dates=dataset_config['date_columns'])

# Data Pipeline

## Define Features and Labels

In [ ]:
# Define the features to include
features = features_config['numerical'] + \
           features_config['categorical']

# Define the labels to include
labels = labels_config

print(f'Features: {features}')
print()
print(f'Labels: {labels}')

## Define Data Preparation Pipeline

In [ ]:
# Instance the data preparation pipeline object
data_preparation = HealthcareDataPreparation(data_transformations_config,
                                             features_config)

In [ ]:
# Get the training data preparation pipeline
data_preparation_pipeline = data_preparation.build_training_data_preparation_pipeline()

In [ ]:
data_preparation_pipeline

## Encode Label

In [ ]:
# Encode the label
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(np.ravel(dataset[labels]))

## Train & Test Split

In [ ]:
# Define X and y for the training set
X = dataset[features]
y = encoded_labels

In [ ]:
# Split training data into train and validation
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=train_test_split_config['test_size'], 
                                                    random_state=train_test_split_config['random_state'])

# Model Training

## Setup Training

In [ ]:
# Set MLflow Experiment
mlflow_experiment_name = model_training_config['mlflow']['experiment_name']

# Set MLflow Experiment
mlflow.set_experiment(mlflow_experiment_name)

In [ ]:
# Initialise trained models dictionary
models = {}

# Initialize DataFrame of models performance
performance = pd.DataFrame(columns=model_training_config['metrics'])

## Logistic Regression

In [ ]:
# Define the model
model_lr = LogisticRegression()

# Create a ModelTrainer
model_trainer_lr = ModelTrainer(model_name=model_training_config['linear_regression']['model_name'], 
                                model=model_lr, 
                                data_pipeline=data_preparation_pipeline)

In [ ]:
model_trainer_lr.pipeline

In [ ]:
# Start an MLflow run
with mlflow.start_run(run_name=model_training_config['linear_regression']['mlflow_run_name']):

    # Fit the model trainer
    model_trainer_lr.bundle_and_fit_pipeline(X_train, y_train)
    
    # Evaluate the model trainer
    evaluation = model_trainer_lr.evaluate_pipeline(X_test, y_test, model_training_config['metrics'])
    
    # Log model's evaluation metrics
    mlflow.log_metrics(evaluation.to_dict()['Value'])
    
    # Log model's features
    mlflow.log_params({'Features': features, 
                       'Labels': labels,
                       'Data Transformations': data_transformations_config,
                       'Model Initial Hyperparameters': None,
                       'Model Optimised Hyperparameters': None})
    
    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=model_trainer_lr.pipeline,
        artifact_path='model_artifacts',
        input_example=X_train.head(1),
        registered_model_name=model_training_config['linear_regression']['mlflow_run_name']
    )